## Example 1: Find misspellings in a file

In [ ]:
import pathlib
import shutil
import subprocess
import tempfile

exe = shutil.which("codespell")
assert exe, "codespell is not on PATH"
d = pathlib.Path(tempfile.mkdtemp())

import importlib.metadata

# Example 1: codespell reports file:line: typo ==> fix, and exits non-zero when it found any
assert importlib.metadata.version("codespell") == "2.4.3"
(d / "notes.txt").write_text("We recieve the data\nand seperate it.\n")
r = subprocess.run([exe, str(d / "notes.txt")], capture_output=True, text=True)
assert r.returncode == 65, (r.returncode, r.stdout, r.stderr)
assert "recieve ==> receive" in r.stdout and "seperate ==> separate" in r.stdout, r.stdout
print("Example 1 OK:\n" + r.stdout)

## Example 2: Fix them in place with -w

In [ ]:
import pathlib
import shutil
import subprocess
import tempfile

exe = shutil.which("codespell")
assert exe, "codespell is not on PATH"
d = pathlib.Path(tempfile.mkdtemp())

# Example 2: --write-changes rewrites unambiguous fixes and leaves the rest alone
f = d / "code.py"
f.write_text("# teh occurence of an adress\nx = 1\n")
r = subprocess.run([exe, "-w", str(f)], capture_output=True, text=True)
assert f.read_text() == "# the occurrence of an address\nx = 1\n", f.read_text()
assert subprocess.run([exe, str(f)], capture_output=True).returncode == 0
print("Example 2 OK:", f.read_text().splitlines()[0])

## Example 3: Ignore words and choose dictionaries

In [ ]:
import pathlib
import shutil
import subprocess
import tempfile

exe = shutil.which("codespell")
assert exe, "codespell is not on PATH"
d = pathlib.Path(tempfile.mkdtemp())

# Example 3: -L keeps project words; --builtin picks the dictionaries (here also the code-oriented one)
f = d / "a.md"
f.write_text("The nd array and the uint8 dtype; teh end.\n")
plain = subprocess.run([exe, "-L", "nd", str(f)], capture_output=True, text=True)
assert "nd ==>" not in plain.stdout and "teh ==> the" in plain.stdout, plain.stdout
code = subprocess.run([exe, "--builtin", "clear,rare,code", "-L", "nd", str(f)], capture_output=True, text=True)
assert "teh ==> the" in code.stdout, code.stdout
print("Example 3 OK")

## Example 4: Options from pyproject.toml

In [ ]:
import pathlib
import shutil
import subprocess
import tempfile

exe = shutil.which("codespell")
assert exe, "codespell is not on PATH"
d = pathlib.Path(tempfile.mkdtemp())

# Example 4: [tool.codespell] in pyproject.toml is read with Python's tomllib (no tomli needed on 3.14)
(d / "pyproject.toml").write_text('[tool.codespell]\nignore-words-list = "recieve"\nskip = "*.log"\n')
(d / "doc.txt").write_text("recieve and seperate\n")
(d / "run.log").write_text("seperate\n")
r = subprocess.run([exe, "--toml", str(d / "pyproject.toml"), str(d)], capture_output=True, text=True, cwd=d)
assert "seperate ==> separate" in r.stdout and "recieve" not in r.stdout and "run.log" not in r.stdout, r.stdout
print("Example 4 OK:", r.stdout.strip())

## Example 5: Encoding detection with chardet (the hard_encoding_detection extra)

In [ ]:
import pathlib
import shutil
import subprocess
import tempfile

exe = shutil.which("codespell")
assert exe, "codespell is not on PATH"
d = pathlib.Path(tempfile.mkdtemp())

import chardet

# Example 5: -e uses chardet to read a file that is not UTF-8 instead of assuming Latin-1
assert chardet.__version__
(d / "legacy.txt").write_bytes("café: teh menu\n".encode("cp1252"))
r = subprocess.run([exe, "-e", str(d / "legacy.txt")], capture_output=True, text=True)
assert "teh ==> the" in r.stdout, (r.stdout, r.stderr)
print("Example 5 OK:", r.stdout.strip())